In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import numpy as np
from idinn.sourcing_model import SingleSourcingModel
from idinn.single_controller import BaseStockController
from idinn.demand import UniformDemand
from idinn.single_controller import SingleSourcingNeuralController
from torch.utils.tensorboard import SummaryWriter
from idinn.sourcing_model import DualSourcingModel
from idinn.dual_controller import CappedDualIndexController
from idinn.dual_controller import DynamicProgrammingController

In [4]:
single_sourcing_model = SingleSourcingModel(
    lead_time=2,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
)
controller_base = BaseStockController()
# z_star should be 11
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 29
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 11.0


tensor(28.8639, grad_fn=<DivBackward0>)

In [5]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=0, high=4),
 )
controller_base = BaseStockController()
# z_star should be 4
controller_base.fit(single_sourcing_model)
print(f"z_star: {controller_base.z_star}")
# Total cost near 10
controller_base.get_total_cost(single_sourcing_model, sourcing_periods=1000).mean()/1000

z_star: 4.0


tensor(10.1073, grad_fn=<DivBackward0>)

In [7]:
single_sourcing_model = SingleSourcingModel(
    lead_time=0,
    holding_cost=5,
    shortage_cost=495,
    batch_size=32,
    init_inventory=10,
    demand_generator=UniformDemand(low=1, high=4),
)

controller_neural = SingleSourcingNeuralController()
controller_neural.fit(
    sourcing_model=single_sourcing_model,
    sourcing_periods=50,
    validation_sourcing_periods=1000,
    epochs=2000,
    tensorboard_writer=SummaryWriter(comment="_single_1"),
    seed=1,
)
controller_neural.get_average_cost(single_sourcing_model, sourcing_periods=1000)

In [7]:
dual_sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)
controller_cdi = CappedDualIndexController()
controller_cdi.fit(
    dual_sourcing_model,
    sourcing_periods=100
)
controller_cdi.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

tensor(26.0550, grad_fn=<DivBackward0>)

In [8]:
controller_cdi.predict(
    current_inventory=3, past_regular_orders=np.array([[1, 1, 1]]), past_expedited_orders=np.array([[0, 0, 0]])
)

(np.int64(2), 0)

In [6]:
dual_sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)
controller_dp = DynamicProgrammingController()
controller_dp.fit(
    dual_sourcing_model,
    max_iterations=10000,
    tolerance=1e-6
)
controller_dp.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

iteration: 100 delta: 0.009350578931446307
iteration: 200 delta: 0.0023492748061642033
iteration: 300 delta: 0.0010458565582283086
iteration: 400 delta: 0.0005887833367026474


IndexError: index -3 is out of bounds for dimension 0 with size 1

In [37]:
controller_dp.get_average_cost(dual_sourcing_model, sourcing_periods=1000)

tensor(24.6350, grad_fn=<DivBackward0>)

In [7]:
current_inventory = dual_sourcing_model.get_current_inventory()
past_regular_orders = dual_sourcing_model.get_past_regular_orders()
past_expedited_orders = dual_sourcing_model.get_past_expedited_orders()

In [85]:
from idinn.demand import CustomDemand

sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=CustomDemand({5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1})
)

In [87]:
controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=1000000,
    tolerance=1e-6
)

AttributeError: 'CustomDemand' object has no attribute 'distribution'

In [20]:
max_demand = 4
min_demand = 0
support = max_demand - min_demand
demand_prob = dict(
    zip(
        np.arange(min_demand, max_demand + 1),
        np.repeat(1.0 / (support + 1.0), int(support + 1)),
    )
)
demand_prob

{0: 0.2, 1: 0.2, 2: 0.2, 3: 0.2, 4: 0.2}

In [80]:
# TODO: Ensure Python >= 3.7 so order is preserved (matches insertion order)
demand_prob = {5: 1., 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0}
# Draw dictionary keys with corresponding probabilities
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)
torch.tensor(list(demand_prob.keys()))[index]


tensor([5, 5, 5, 5, 5, 5, 5, 5, 5, 5])

In [34]:
demand_prob = {5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1}
# Draw dictionary keys with corresponding probabilities
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)
torch.tensor(list(demand_prob.keys()))[index]



# Use torch multinomial to sample from dictionary demand_prob
index = torch.multinomial(torch.tensor(list(demand_prob.values())), num_samples=10, replacement=True)

torch.tensor(list(demand_prob.keys()))[index]

tensor([0, 4, 2, 3, 4, 2, 2, 1, 3, 2])

In [35]:
demand_prob.values()

dict_values([0.2, 0.2, 0.2, 0.2, 0.2])

In [84]:


demand_prob = {5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1}
custom_demand = CustomDemand(demand_prob)
custom_demand.sample(100, 3)
custom_demand.enumerate_support()

{5: 0.2, 6: 0.3, 7: 0.1, 8: 0.1, 9: 0.1}

In [10]:
controller.predict(current_inventory=2, past_regular_orders=[1, 1, 1])

(3, 1)

In [10]:
current_inventory = 10
past_regular_orders = [1, 2, 3, 4, 5]
past_expedited_orders = [6, 7, 8]
regular_lead_time = 3



(13, 4, 5)


In [21]:
import numpy as np
current_inventory = 10
regular_lead_time = 5
expedited_lead_time = 2
past_regular_orders = np.array([1, 2, 3, 4, 5, 6, 7, 8])
past_expedited_orders = np.array([6, 7, 8])
# Output:
past_orders = [4, 5, 6, 7, 8]

# Pad with zeros for past_expedited_orders
# e_2 = np.pad(e_2, (0, 3), 'constant')
# Start with np.zeros before padding
e_1 = np.zeros(regular_lead_time)
e_1 = past_regular_orders[-regular_lead_time:]

e_2 = np.zeros(regular_lead_time)
e_2[:expedited_lead_time] = past_expedited_orders[-expedited_lead_time:]
e_3 = e_1 + e_2

first = current_inventory + e_3[0]
second = e_3[-regular_lead_time+1:]
print(tuple([first]+second))

(34.0, 27.0, 28.0, 29.0)


In [ ]:
first =  past_regular_orders[-regular_lead_time]
second = past_regular_orders[-regular_lead_time+1:]
print(tuple([first]+second))

In [ ]:
controller = CappedDualIndexController()
controller.fit(sourcing_model, sourcing_periods=1000)
controller.predict(
    current_inventory,
    past_regular_orders,
    past_expedited_orders
)

In [3]:
sourcing_model = DualSourcingModel(
    regular_lead_time=3,
    expedited_lead_time=0,
    regular_order_cost=0,
    expedited_order_cost=20,
    holding_cost=5,
    shortage_cost=495,
    init_inventory=0,
    demand_generator=UniformDemand(low=0, high=4)
)

controller = DynamicProgrammingController()

controller.fit(
    sourcing_model,
    max_iterations=10000,
    tolerance=1
)